# Qwen2.5-1.5B 한·영 64:64 혼합 보정 실험

예비 실험의 **같은 표본**을 사용해 혼합 보정 GPTQ 4bit 모델 하나만 새로 만듭니다. Colab GPU(T4) 런타임에서 위에서 아래로 실행하세요. Google Drive 연결은 필요하지 않습니다. 모델 체크포인트는 Colab 런타임에만 남고, 결과 CSV와 실행 정보는 ZIP으로 컴퓨터에 다운로드합니다.

In [ ]:
%pip install -q "gptqmodel==7.10.0"

In [ ]:
import gc
import hashlib
import io
import json
import math
import random
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import torch
import transformers
import gptqmodel
from google.colab import files
from transformers import AutoTokenizer
from gptqmodel import BACKEND, GPTQModel, GPTQConfig

if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임을 GPU로 바꾸고 다시 실행하세요.")
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "Transformers:", transformers.__version__,
      "GPTQModel:", gptqmodel.__version__)

## 1. 예비 실험 파일 업로드

파일 선택 창에서 로컬 `pilot_results` 폴더의 **`pilot_samples.json`과 `ppl_by_group.csv` 두 파일**을 함께 선택하세요. 다른 표본이나 기존 결과를 새로 만들지 않습니다.

In [ ]:
uploaded = files.upload()
required_files = {"pilot_samples.json", "ppl_by_group.csv"}
if set(uploaded) != required_files:
    raise ValueError(f"정확히 두 파일을 업로드하세요: {sorted(required_files)}")

samples = json.loads(uploaded["pilot_samples.json"].decode("utf-8"))
if len(samples.get("calibration_en", [])) != 128 or len(samples.get("calibration_ko", [])) != 128:
    raise ValueError("예비 실험의 영어·한국어 보정 문서가 각각 128개여야 합니다.")
if set(samples.get("eval_groups", {})) != {"ko", "en"} or any(
    len(samples["eval_groups"][lang]) != 3 for lang in ("ko", "en")
):
    raise ValueError("한국어·영어 평가 묶음이 각각 3개여야 합니다.")

baseline = pd.read_csv(io.BytesIO(uploaded["ppl_by_group.csv"]))
expected_conditions = {"FP16", "RTN 4bit", "GPTQ English 4bit", "GPTQ Korean 4bit"}
expected_keys = {(condition, lang, group) for condition in expected_conditions
                 for lang in ("ko", "en") for group in (1, 2, 3)}
required_columns = {"condition", "language", "group", "ppl", "predicted_tokens", "D_percent"}
if not required_columns.issubset(baseline.columns) or len(baseline) != 24 or set(
    zip(baseline["condition"], baseline["language"], baseline["group"])
) != expected_keys:
    raise ValueError("예비 실험 CSV에 4조건 × 2언어 × 3묶음의 고유 결과 24행이 필요합니다.")
if baseline["ppl"].isna().any() or (baseline["ppl"] <= 0).any():
    raise ValueError("예비 실험 PPL 값이 유효하지 않습니다.")
print("표본 확인: 보정 128개씩, 평가 3묶음씩; 기준 결과 24행")

## 2. 같은 평가 표본 준비

예비 실험과 동일하게 각 문서의 토큰을 이어 붙이고 EOS를 더한 뒤, 언어별 각 묶음에서 8192토큰을 사용합니다.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
CALIBRATION_LENGTH = 512
EVAL_SEQUENCE_LENGTH = 512
EVAL_TOKENS_PER_GROUP = 16 * EVAL_SEQUENCE_LENGTH
CONDITION = "GPTQ Mixed 64:64 4bit"
OUTPUT_DIR = Path("/content/Team22/mixed_64_64")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

def make_token_groups(text_groups):
    token_groups = []
    for texts in text_groups:
        ids = []
        for text in texts:
            ids.extend(tokenizer(text, add_special_tokens=False)["input_ids"])
            if tokenizer.eos_token_id is not None:
                ids.append(tokenizer.eos_token_id)
            if len(ids) >= EVAL_TOKENS_PER_GROUP:
                break
        ids = ids[:EVAL_TOKENS_PER_GROUP]
        if len(ids) != EVAL_TOKENS_PER_GROUP:
            raise ValueError("평가 묶음의 토큰 수가 부족합니다.")
        token_groups.append(ids)
    return token_groups

eval_token_groups = {lang: make_token_groups(samples["eval_groups"][lang])
                     for lang in ("ko", "en")}
mixed_texts = samples["calibration_en"][:64] + samples["calibration_ko"][:64]
random.Random(22).shuffle(mixed_texts)
assert len(mixed_texts) == 128
print("혼합 보정: 영어 64개 + 한국어 64개; 평가: 언어별 3 × 8192토큰")

## 3. 혼합 보정 모델 하나만 양자화

예비 실험과 같은 GPTQ 설정과 `GPTQ_TORCH` 백엔드를 사용합니다. 같은 Colab 런타임에 완전한 체크포인트가 있으면 재사용합니다.

In [ ]:
pool_globals = gptqmodel.get_device_thread_pool.__globals__
pool_globals["_default_cpu_workers"] = lambda: 2
checkpoint = OUTPUT_DIR / "gptq_mixed_4bit"
weights_exist = any(checkpoint.glob("*.safetensors")) or any(checkpoint.glob("pytorch_model*.bin"))
if (checkpoint / "config.json").exists() and weights_exist:
    print("기존 체크포인트 사용:", checkpoint)
else:
    if checkpoint.exists() and any(checkpoint.iterdir()):
        raise RuntimeError(f"불완전한 체크포인트를 확인한 뒤 정리하세요: {checkpoint}")
    model = GPTQModel.load(
        MODEL_ID,
        quantize_config=GPTQConfig(bits=4, group_size=128, sym=True, desc_act=False),
        device="cuda:0", dtype=torch.float16,
    )
    calibration = [{"input_ids": tokenizer(text, truncation=True,
                    max_length=CALIBRATION_LENGTH)["input_ids"]} for text in mixed_texts]
    model.quantize(calibration, batch_size=1, backend=BACKEND.GPTQ_TORCH)
    model.save(str(checkpoint))
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("혼합 보정 체크포인트 저장:", checkpoint)

## 4. PPL 평가와 기존 결과 비교

한국어·영어 각 3묶음의 PPL을 계산합니다. 저하율은 저장된 예비 실험 CSV의 **같은 언어·같은 묶음 FP16**을 기준으로 계산합니다.

In [ ]:
model = GPTQModel.load(str(checkpoint), device="cuda:0", dtype=torch.float16,
                       backend=BACKEND.GPTQ_TORCH)
model.eval()
rows = []
for lang, groups in eval_token_groups.items():
    for group_idx, token_ids in enumerate(groups, start=1):
        total_nll = 0.0
        total_predicted_tokens = 0
        for start in range(0, len(token_ids), EVAL_SEQUENCE_LENGTH):
            chunk = token_ids[start:start + EVAL_SEQUENCE_LENGTH]
            input_ids = torch.tensor([chunk], dtype=torch.long, device="cuda:0")
            with torch.inference_mode():
                output = model(input_ids=input_ids, labels=input_ids)
            n_predicted = len(chunk) - 1
            total_nll += float(output.loss.detach().float().cpu()) * n_predicted
            total_predicted_tokens += n_predicted
            del input_ids, output
        mean_nll = total_nll / total_predicted_tokens
        rows.append({"condition": CONDITION, "language": lang, "group": group_idx,
                     "mean_nll": mean_nll, "ppl": math.exp(mean_nll),
                     "predicted_tokens": total_predicted_tokens})
del model
gc.collect()
torch.cuda.empty_cache()

mixed = pd.DataFrame(rows)
fp16 = baseline[baseline["condition"] == "FP16"].set_index(["language", "group"])
for row in rows:
    key = (row["language"], row["group"])
    if row["predicted_tokens"] != fp16.loc[key, "predicted_tokens"]:
        raise ValueError(f"평가 토큰 수가 기존 FP16과 다릅니다: {key}")
mixed["D_percent"] = mixed.apply(
    lambda row: (row["ppl"] / fp16.loc[(row["language"], row["group"]), "ppl"] - 1) * 100,
    axis=1,
)
comparison = pd.concat([
    baseline[baseline["condition"].isin(
        ["FP16", "GPTQ English 4bit", "GPTQ Korean 4bit"])], mixed
], ignore_index=True)
summary = comparison.groupby(["condition", "language"], as_index=False).agg(
    mean_ppl=("ppl", "mean"), mean_D_percent=("D_percent", "mean")
)
print("혼합 보정 묶음별 결과")
display(mixed)
print("기존 보정 방식과 비교 (언어별 평균)")
display(summary)

## 5. 결과 다운로드

ZIP 다운로드가 완료됐는지 브라우저에서 확인하세요. ZIP에는 모델 체크포인트가 들어 있지 않습니다.

In [ ]:
mixed.to_csv(OUTPUT_DIR / "mixed_ppl_by_group.csv", index=False)
summary.to_csv(OUTPUT_DIR / "comparison_summary.csv", index=False)
run_info = {
    "model_id": MODEL_ID, "condition": CONDITION, "seed": 22,
    "calibration_en": 64, "calibration_ko": 64,
    "gptq": {"bits": 4, "group_size": 128, "sym": True, "desc_act": False},
    "torch": torch.__version__, "transformers": transformers.__version__,
    "gptqmodel": gptqmodel.__version__, "gpu": torch.cuda.get_device_name(0),
    "pilot_samples_sha256": hashlib.sha256(uploaded["pilot_samples.json"]).hexdigest(),
    "pilot_ppl_sha256": hashlib.sha256(uploaded["ppl_by_group.csv"]).hexdigest(),
}
(OUTPUT_DIR / "mixed_runtime.json").write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8"
)
archive = OUTPUT_DIR / "mixed_64_64_results.zip"
with ZipFile(archive, "w") as bundle:
    for name in ("mixed_ppl_by_group.csv", "comparison_summary.csv", "mixed_runtime.json"):
        bundle.write(OUTPUT_DIR / name, arcname=name)
print("다운로드할 결과:", archive)
files.download(str(archive))